# Color Calibration

Color calibration standardizes color appearance across recordings made under different lighting
conditions, with different cameras, or across sessions. Two scenarios arise in ecological
imaging: correcting **still images** (morphometric photographs, drone stills) and correcting
**video recordings** (behavioral experiments, continuous monitoring) — the focus of
*An Imaging Primer for Ecologists* (Francisco et al., 2026).

**What you need:** a color reference target — a neutral gray card (white balance only) or a
24-patch color checker (full correction). Hold it in front of the camera under the same lighting
used for the experiment at the start of each session.

> Set your camera to a **fixed white balance** before recording. Adaptive white balance changes
> the color balance frame to frame, making a single session-wide correction unreliable. GoPros default to
> adaptive — disable this before each session.

This notebook has two parts:

1. **Software walkthroughs** (below) — step-by-step GUI guides for Lightroom, GIMP,
   DaVinci Resolve, Final Cut Pro, and ffmpeg. Links to illustrated official documentation accompany the guides.
2. **Python / OpenCV** — a runnable implementation that computes white-balance gains and a
   full Color Correction Matrix, then applies them to a video.


## Still images — Lightroom

1. Import the reference-target photo and open the Edit panel.
2. Use the white-balance selector on an unclipped neutral patch under the recording illumination.
3. Copy the white-balance edit settings and paste them onto images taken with the same fixed camera settings and lighting.
4. Export with a consistent color profile and retain the originals and edit settings.

See [Adobe's illustrated Lightroom desktop guide](https://helpx.adobe.com/lightroom/desktop/edit-photos/edit-photos.html). White balance removes a neutral color cast; it is not a full radiometric or spectral calibration.


## Still images — GIMP

Open the reference-target photograph and use **Colors → Levels** to set a neutral gray reference. Inspect the result and apply consistent settings to images from the same acquisition conditions. Curves can adjust individual channels, but avoid clipping and retain the original data.

See the illustrated official guides for [Levels](https://docs.gimp.org/3.0/en/gimp-tool-levels.html) and [Curves](https://docs.gimp.org/3.0/en/gimp-tool-curves.html). Batch interfaces differ by GIMP version; the Python video example below provides an executable alternative.


## Video — general approach

1. **Film the reference target** in the first few seconds of each recording, under the same
   lighting as the experiment.
2. **Extract the reference frame** from the video.
3. **Compute the correction** from that frame.
4. **Apply to the whole clip** — DaVinci and Final Cut apply the grade to the entire clip
   automatically; ffmpeg and the Python code below process the video frame by frame.

To extract the reference frame with ffmpeg:

```bash
# Extract frame at timestamp 00:00:02 (adjust to where the target appears)
ffmpeg -ss 00:00:02 -i input.mp4 -frames:v 1 reference_frame.png
```


## Video — DaVinci Resolve

Import a clip, open the Color page, and locate the recorded reference target. Adjust the grade using the neutral patches and scopes; if using chart matching, select the matching chart type and align its overlay with the target. Apply the same grade only to clips recorded under matching lighting and camera settings. Export the corrected clip from the Deliver page.

Use the [official Resolve training guides](https://www.blackmagicdesign.com/products/davinciresolve/training), including the Colorist Guide, for illustrated version-specific controls. Grading is non-destructive inside the project; exporting the corrected video renders a new file.


## Video — Final Cut Pro

Select the clip and enable Balance Color. In the Video inspector, change the Balance Color method to White Balance and sample an appropriate neutral reference in the recorded target. Inspect the result before applying matching corrections to other clips and exporting.

See Apple's illustrated guide to [manual white balance](https://support.apple.com/guide/final-cut-pro/manually-white-balance-clips-ver691a72df7/mac). Match the instructions to your installed version and retain the original clips.


## Video — ffmpeg (command line)

ffmpeg can apply color corrections to a video in a single pass without an editing GUI.
The most practical approach is to compute per-channel gains or a Color Correction Matrix with
the Python code below and pass them directly to ffmpeg's `colorchannelmixer` filter.

**Simple white balance — per-channel gain:**

```bash
# Replace the gains with values printed by the white-balance cell below
ffmpeg -i input.mp4 \
  -vf "colorchannelmixer=rr=0.94:gg=1.0:bb=1.12" \
  -c:v libx264 -crf 18 -preset fast \
  output_corrected.mp4
```

**Full 3x3 color correction matrix (CCM):**

```bash
# colorchannelmixer accepts all 9 matrix entries
# rr rg rb = how much R, G, B contribute to the output Red channel, etc.
ffmpeg -i input.mp4 \
  -vf "colorchannelmixer=rr=0.95:rg=0.03:rb=0.02:gr=0.01:gg=1.01:gb=-0.02:br=-0.01:bg=0.04:bb=1.08" \
  -c:v libx264 -crf 18 -preset fast \
  output_corrected.mp4
```

**Batch-process a folder of videos:**

```bash
mkdir -p ./corrected
for f in ./recordings/*.mp4; do
  ffmpeg -i "$f" \
    -vf "colorchannelmixer=rr=0.94:gg=1.0:bb=1.12" \
    -c:v libx264 -crf 18 -preset fast \
    "./corrected/$(basename "$f")"
done
```

The matrix below uses row-vector multiplication (`RGB @ ccm`). FFmpeg lists coefficients by output channel: use the **columns** of `ccm` for `rr,rg,rb`, then `gr,gg,gb`, then `br,bg,bb`. The example numbers above are illustrative, not a calibration for your camera.


## Python / OpenCV

This example compares per-channel white-balance gains with a 3×3 least-squares color correction matrix (CCM), then processes a short generated sample video. No external recording is required for Run All.

A 3×3 matrix models linear channel mixing; it cannot correct arbitrary nonlinear responses or recover clipped channels. This demonstration fits encoded RGB values for accessibility. Quantitative color work requires an appropriate color-space/transfer-function workflow, target-specific reference values under a specified illuminant, and independent validation. A printed illustration is not a calibrated physical color target.

Install the repository's `requirements.txt` first. To process real video, supply a recording and manually measured patch centers in the final cell.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

### Example ColorChecker reference values

The following 24 approximate RGB triplets are illustrative targets for this synthetic demonstration. They are not a certificate for a physical chart. For real measurements use the manufacturer's reference data for your target, illuminant, and working color space. Neutral patches are approximately, rather than exactly, equal in R, G, and B.


In [ ]:
# Illustrative ColorChecker-like RGB values (0-255); see limitations above.
# Row-major order matching the physical 4-row × 6-column layout.
COLORCHECKER_REF = np.array([
    [115,  82,  68],   #  1  Dark skin
    [194, 150, 130],   #  2  Light skin
    [ 98, 122, 157],   #  3  Blue sky
    [ 87, 108,  67],   #  4  Foliage
    [133, 128, 177],   #  5  Blue flower
    [103, 189, 170],   #  6  Bluish green
    [214, 126,  44],   #  7  Orange
    [ 80,  91, 166],   #  8  Purplish blue
    [193,  90,  99],   #  9  Moderate red
    [ 94,  60, 108],   # 10  Purple
    [157, 188,  64],   # 11  Yellow green
    [224, 163,  46],   # 12  Orange yellow
    [ 56,  61, 150],   # 13  Blue
    [ 70, 148,  73],   # 14  Green
    [175,  54,  60],   # 15  Red
    [231, 199,  31],   # 16  Yellow
    [187,  86, 149],   # 17  Magenta
    [  8, 133, 161],   # 18  Cyan
    [243, 243, 242],   # 19  White
    [200, 200, 200],   # 20  Neutral 8
    [160, 160, 160],   # 21  Neutral 6.5
    [122, 122, 121],   # 22  Neutral 5   <- used for white balance
    [ 85,  85,  85],   # 23  Neutral 3.5
    [ 52,  52,  52],   # 24  Black
], dtype=np.float32)

print(f"Loaded {len(COLORCHECKER_REF)} ColorChecker patches.")
print(f"Neutral patch (index 21, Neutral 5): RGB = {COLORCHECKER_REF[21]}")


### Create a synthetic test image

Generate a 4×6 grid and apply an artificial warm color cast. The boosted red channel clips some bright patches, illustrating information that cannot be recovered by a correction matrix. Real images need measured patch centers; the synthetic chart coordinates must not be reused on arbitrary frames.


In [ ]:
N_COLS, N_ROWS = 6, 4
PATCH_PX  = 80   # patch size in pixels
BORDER_PX =  6   # border between patches


def make_synthetic_colorchecker(color_cast=(1.0, 1.0, 1.0)):
    """Return a BGR image of a synthetic ColorChecker with an optional color cast."""
    W = N_COLS * PATCH_PX + (N_COLS + 1) * BORDER_PX
    H = N_ROWS * PATCH_PX + (N_ROWS + 1) * BORDER_PX
    img = np.zeros((H, W, 3), dtype=np.float32)

    cast = np.array(color_cast, dtype=np.float32)

    for i, ref_rgb in enumerate(COLORCHECKER_REF):
        row, col = divmod(i, N_COLS)
        y = BORDER_PX + row * (PATCH_PX + BORDER_PX)
        x = BORDER_PX + col * (PATCH_PX + BORDER_PX)
        bgr = np.clip(ref_rgb * cast, 0, 255)[::-1]
        img[y:y + PATCH_PX, x:x + PATCH_PX] = bgr

    return img.astype(np.uint8)


def synthetic_patch_centers():
    return np.array([
        [BORDER_PX + (i % N_COLS) * (PATCH_PX + BORDER_PX) + PATCH_PX / 2,
         BORDER_PX + (i // N_COLS) * (PATCH_PX + BORDER_PX) + PATCH_PX / 2]
        for i in range(24)
    ], dtype=float)


def sample_patch(img_bgr, patch_idx, patch_centers=None, radius=12):
    """Mean BGR in an explicitly located patch, away from its borders."""
    centers = synthetic_patch_centers() if patch_centers is None else np.asarray(patch_centers, dtype=float)
    if centers.shape != (24, 2) or not np.isfinite(centers).all():
        raise ValueError("Provide 24 finite (x, y) centers in chart row-major order.")
    if not isinstance(radius, int) or radius < 1:
        raise ValueError("radius must be a positive integer.")
    x, y = np.rint(centers[patch_idx]).astype(int)
    h, w = img_bgr.shape[:2]
    if x-radius < 0 or y-radius < 0 or x+radius >= w or y+radius >= h:
        raise ValueError(f"Patch {patch_idx + 1} sampling region lies outside the image.")
    return img_bgr[y-radius:y+radius+1, x-radius:x+radius+1].mean(axis=(0, 1)).astype(np.float32)


# Warm cast: red boosted ×1.35, blue reduced ×0.72
# (mimics incandescent light or shallow underwater haze)
img_cast = make_synthetic_colorchecker(color_cast=(1.35, 1.0, 0.72))
img_ref  = make_synthetic_colorchecker(color_cast=(1.0,  1.0, 1.0))

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].imshow(cv2.cvtColor(img_ref,  cv2.COLOR_BGR2RGB))
axs[0].set_title("Reference (no cast)")
axs[0].axis("off")
axs[1].imshow(cv2.cvtColor(img_cast, cv2.COLOR_BGR2RGB))
axs[1].set_title("With warm color cast (R×1.35, B×0.72)")
axs[1].axis("off")
plt.suptitle("Synthetic ColorChecker — before correction", fontsize=13)
plt.tight_layout()
plt.show()


### Method 1 — White balance using the neutral gray patch

We sample patch 22 (Neutral 5, reference sRGB ~ 122, 122, 121) from the cast image.
Per-channel gain factors are computed so the sampled color matches the reference, then applied
to every pixel. This is equivalent to clicking the gray patch with the eyedropper in Lightroom
or DaVinci. The printed gains map directly to ffmpeg `colorchannelmixer=rr=..:gg=..:bb=..`.


In [ ]:
def compute_wb_gains(img_bgr, gray_patch_idx=21, patch_centers=None, radius=12):
    """Per-channel gains to neutralize the chosen gray patch."""
    measured_bgr = sample_patch(img_bgr, gray_patch_idx, patch_centers, radius)
    ref_rgb      = COLORCHECKER_REF[gray_patch_idx]
    ref_bgr      = ref_rgb[::-1]
    return ref_bgr / (measured_bgr + 1e-6)


def apply_gains(img_bgr, gains):
    out = img_bgr.astype(np.float32) * gains[np.newaxis, np.newaxis, :]
    return np.clip(out, 0, 255).astype(np.uint8)


wb_gains = compute_wb_gains(img_cast, gray_patch_idx=21)
img_wb   = apply_gains(img_cast, wb_gains)

print(f"White balance gains  B={wb_gains[0]:.3f}  G={wb_gains[1]:.3f}  R={wb_gains[2]:.3f}")
print(f"ffmpeg:  colorchannelmixer=rr={wb_gains[2]:.3f}:gg={wb_gains[1]:.3f}:bb={wb_gains[0]:.3f}")

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for ax, img, title in zip(axs, [img_cast, img_wb, img_ref],
                           ["Cast", "After white balance", "Reference (ground truth)"]):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


### Method 2 — Color Correction Matrix (CCM)

A 3x3 CCM simultaneously maps all 24 measured patch colors toward their reference values using
**least-squares linear regression**:

$$\mathbf{M}_{\text{measured}} \cdot \mathbf{CCM} \approx \mathbf{R}_{\text{reference}}$$

where **M** is the 24x3 matrix of measured RGB values and **R** the 24x3 matrix of reference
values. This handles cross-channel contamination and is more accurate than white balance alone.


In [ ]:
def measure_all_patches(img_bgr, patch_centers=None, radius=12):
    """Sample all 24 patches; return (24, 3) float32 array in RGB order."""
    return np.array([sample_patch(img_bgr, i, patch_centers, radius)[::-1] for i in range(len(COLORCHECKER_REF))],
                    dtype=np.float32)


def compute_ccm(img_bgr, patch_centers=None, radius=12):
    M = measure_all_patches(img_bgr, patch_centers, radius)
    if np.linalg.matrix_rank(M) < 3:
        raise ValueError("Patch colors cannot determine a full-rank CCM; check target selection.")
    ccm, _, _, _ = np.linalg.lstsq(M, COLORCHECKER_REF, rcond=None)
    return ccm.astype(np.float32)


def apply_ccm(img_bgr, ccm):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    corrected_rgb = np.clip(img_rgb @ ccm, 0, 255).astype(np.uint8)
    return cv2.cvtColor(corrected_rgb, cv2.COLOR_RGB2BGR)


ccm     = compute_ccm(img_cast)
img_ccm = apply_ccm(img_cast, ccm)

print("Color Correction Matrix (maps measured RGB -> reference RGB):")
print(ccm.round(4))
print()
print("A perfect CCM for a linear color cast is diagonal.")
print("Off-diagonal terms correct cross-channel contamination.")


### Evaluate: per-patch error before and after correction

Mean absolute error (MAE) is measured here in encoded RGB units, not a perceptual color-difference metric. The guide line at 5 units is illustrative, not a universal ecological acceptance threshold. These are fitting-patch errors; assess a real calibration against independent reference data and requirements of the biological measurement.


In [ ]:
def patch_errors(img_bgr):
    return np.abs(measure_all_patches(img_bgr) - COLORCHECKER_REF).mean(axis=1)


err_cast = patch_errors(img_cast)
err_wb   = patch_errors(img_wb)
err_ccm  = patch_errors(img_ccm)

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for ax, img, errs, title in zip(
    axs,
    [img_cast, img_wb, img_ccm],
    [err_cast, err_wb, err_ccm],
    ["Original (warm cast)", "White balance", "CCM corrected"],
):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title}\nMean error: {errs.mean():.1f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

patch_ids = np.arange(1, 25)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(patch_ids, err_cast, "o--", color="C1", label=f"Cast    mean={err_cast.mean():.1f}")
ax.plot(patch_ids, err_wb,   "s--", color="C2", label=f"WB      mean={err_wb.mean():.1f}")
ax.plot(patch_ids, err_ccm,  "^-",  color="C0", label=f"CCM     mean={err_ccm.mean():.1f}")
ax.axhline(5, color="gray", ls=":", lw=1, label="Error=5 threshold")
ax.set_xlabel("Patch number (1-24)")
ax.set_ylabel("Mean absolute error (RGB, 0-255)")
ax.set_title("Per-patch color error before and after correction")
ax.legend()
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()

### Apply the correction to a video recording

By default this cell generates and processes a short synthetic video in `notebook_outputs/`. It exercises the same video-reading, fitting, and writing path used for real recordings.

For real data, set `USE_SYNTHETIC_VIDEO = False`, provide `INPUT_VIDEO` and `REF_FRAME_IDX`, and measure all 24 `PATCH_CENTERS` in that frame. Use chart row-major order (dark skin first, black last), regardless of the chart's orientation in the image. Choose `PATCH_RADIUS` so every sampled square lies inside its patch. Inspect the numbered overlay before trusting the result; charts at steep angles need small regions or prior rectification. Keep acquisition settings and illumination fixed.

The demo processes 8-bit BGR video and writes video only (no audio). It is not a RAW/log/HDR calibration pipeline. Outputs are derivatives; originals are retained.


In [ ]:
USE_SYNTHETIC_VIDEO = True
INPUT_VIDEO = "recording.mp4"  # used only when USE_SYNTHETIC_VIDEO is False
REF_FRAME_IDX = 0
PATCH_CENTERS = None  # real data: 24 measured [x, y] pairs in chart row-major order
PATCH_RADIUS = 12
output_dir = Path("notebook_outputs")
output_dir.mkdir(exist_ok=True)
OUTPUT_VIDEO = output_dir / "recording_corrected.mp4"

if USE_SYNTHETIC_VIDEO:
    INPUT_VIDEO = output_dir / "synthetic_colorchecker.avi"
    PATCH_CENTERS = synthetic_patch_centers()
    REF_FRAME_IDX = 0
    h, w = img_cast.shape[:2]
    sample_writer = cv2.VideoWriter(str(INPUT_VIDEO), cv2.VideoWriter_fourcc(*"MJPG"), 10, (w, h))
    if not sample_writer.isOpened():
        raise RuntimeError("Could not create sample video; check OpenCV codec support.")
    try:
        for _ in range(12):
            sample_writer.write(img_cast)
    finally:
        sample_writer.release()


def correct_video(input_path, output_path, patch_centers, ref_frame_idx=0, radius=12):
    src, dst = Path(input_path), Path(output_path)
    if src.resolve() == dst.resolve():
        raise ValueError("Input and output paths must differ.")
    if not src.is_file():
        raise FileNotFoundError(f"Input video not found: {src.resolve()}")
    if patch_centers is None:
        raise ValueError("Real video requires explicitly measured PATCH_CENTERS.")
    if not isinstance(ref_frame_idx, int) or ref_frame_idx < 0:
        raise ValueError("REF_FRAME_IDX must be a non-negative integer.")
    cap, writer = cv2.VideoCapture(str(src)), None
    try:
        if not cap.isOpened():
            raise RuntimeError(f"Could not open {src}.")
        cap.set(cv2.CAP_PROP_POS_FRAMES, ref_frame_idx)
        ok, reference = cap.read()
        if not ok:
            raise ValueError(f"Cannot read reference frame {ref_frame_idx}.")
        matrix = compute_ccm(reference, patch_centers, radius)
        plt.figure(figsize=(10, 6))
        plt.imshow(cv2.cvtColor(reference, cv2.COLOR_BGR2RGB))
        for i, (x, y) in enumerate(patch_centers, 1):
            plt.plot([x-radius, x+radius, x+radius, x-radius, x-radius],
                     [y-radius, y-radius, y+radius, y+radius, y-radius], color="white")
            plt.text(x, y, str(i), color="white", ha="center")
        plt.title("Check patch sampling regions against the reference target")
        plt.axis("off")
        plt.show()
        fps = cap.get(cv2.CAP_PROP_FPS)
        if not np.isfinite(fps) or fps <= 0:
            raise ValueError("Video frame rate is unavailable; specify a verified frame rate.")
        h, w = reference.shape[:2]
        if h % 2 or w % 2:
            raise ValueError("This MP4 example requires even frame dimensions to avoid codec cropping.")
        dst.parent.mkdir(parents=True, exist_ok=True)
        writer = cv2.VideoWriter(str(dst), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
        if not writer.isOpened():
            raise RuntimeError(f"Could not create {dst}.")
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        n = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            writer.write(apply_ccm(frame, matrix))
            n += 1
        if not n:
            raise RuntimeError("No frames were processed.")
        return n, matrix
    finally:
        cap.release()
        if writer is not None:
            writer.release()


n_written, video_ccm = correct_video(INPUT_VIDEO, OUTPUT_VIDEO, PATCH_CENTERS, REF_FRAME_IDX, PATCH_RADIUS)
print(f"Wrote {n_written} corrected frames to {OUTPUT_VIDEO.resolve()}")


## Tips

- **One reference shot per lighting condition**, not just per session. Re-film the target
  whenever the light changes (moving indoors/outdoors, time of day, cloud cover).
- **Avoid glare** on the reference target: tilt it slightly (less than 10 degrees) relative to
  the main light source.
- **Underwater:** red light is absorbed rapidly with depth, producing a strong blue-green cast.
  Take a reference shot at the start of each dive. For physically based underwater color
  restoration see [Sea-Thru](https://github.com/hainh/sea-thru) (Akkaynak & Treibitz 2019).
- **Export color space:** use sRGB for display/tracking pipelines, AdobeRGB or ProPhoto RGB
  for quantitative color analysis. Mixing color spaces across clips invalidates comparisons.
- **No re-encoding:** the gains and CCM printed above can be passed straight to ffmpeg's
  `colorchannelmixer` filter (see the ffmpeg section) to apply the correction in one pass.
